# Comparative evaluation of temperature-based solar-radiation models using a Global Performance Indicator

**Authoritative analysis notebook.** Run this notebook from top to bottom with **Kernel → Restart Kernel and Run All**. It reads the three retained IMD files and rebuilds every scientific result and figure. No `.py` analysis runner is required.

## Study design

- Stations: **Ahmedabad, Amreli, Okha**.
- Periods: **Annual, Winter, Pre-Monsoon, Monsoon, Post-Monsoon**.
- Models: **M1–M16**, defined once in the model-registry cell below and exported to `results/model_registry.csv`.
- Source files cover 1985–2025. Each station contributes its own valid dates; there is no forced common period.
- Target output: exactly **3 × 5 × 16 = 240** station-period-model combinations.

The reference is a sunshine-derived estimate, not pyranometer ground truth. Results therefore assess agreement with the reference method, not absolute irradiance accuracy.


## Research source and reference radiation

Karale, Misra, Ghosh & Latwal (2026), *Seasonal and regional evaluation of sixteen temperature-based solar radiation models for India using a Global Performance Indicator*, supplied preprint, is the primary method source. Its Ångström–Prescott formulation is Eq. 5, p. 5:

\[R_{s,AP} = (a+b\,n/N)R_a,\qquad a=0.25,\quad b=0.50.\]

`n` is observed bright-sunshine duration in **hours**; `N=24ω_s/π` is maximum possible astronomical sunshine duration in **hours**; `ω_s` is sunset hour angle in radians; and `R_a` is extraterrestrial radiation in **MJ m⁻² day⁻¹**. The calculation needs station latitude, day of year, sunshine duration, and solar geometry. Tmax/Tmin do not enter this reference equation. Rows with missing sunshine or `n` outside `[0,N]` are excluded.

A supplied companion report is retained as background but is not a primary paper: it conflicts with Table 2 on M2 notation and describes a different study scope, so it is not used to define this project's scope or model equations.

The same paper's solar-geometry Eq. 2, p. 5, explicitly prints `dr=1+0.003 cos(2πJ/365)`. This notebook uses **0.003** for the primary results. FAO-56 commonly uses 0.033; that alternative is retained as a sensitivity check, not substituted silently. The paper's printed unit for `Gsc=0.0820` appears inconsistent with the `24×60` conversion in the extraterrestrial-radiation equation; the implementation uses the conventional unit MJ m⁻² min⁻¹ and records this as a unit interpretation.

## Inputs and quality rules

Only `data/raw_imd/AHMEDABAD.csv`, `AMRELI.csv`, and `OKHA.csv` are analysis inputs. Required fields are `YEAR`, `MN`, `DT`, `MAX`, `MIN`, `SSH`. Dates must parse and be unique. Values are numerically coerced; missing values remain missing. A record is included only if Tmax/Tmin and sunshine are present, Tmax>Tmin, `0≤SSH≤N`, and every model prediction is finite. No observations are imputed, shifted or clipped. QC and provenance exports report counts and station-specific coverage.


## 1. Imports, configuration and project discovery

Project paths are resolved relative to the notebook kernel working directory; no machine-specific paths are embedded.

In [1]:
from pathlib import Path
import hashlib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "data" / "raw_imd").is_dir() and (directory / "Research_References").is_dir():
            return directory
    raise FileNotFoundError("Could not locate project root containing data/raw_imd and Research_References")

ROOT = find_project_root()
RESULTS = ROOT / "results"
FINAL = RESULTS / "final"
QC = RESULTS / "qc"
VALIDATION = RESULTS / "validation"
FIGURES = ROOT / "figures" / "final"
STATIONS = ["Ahmedabad", "Amreli", "Okha"]
LATITUDE = {"Ahmedabad": 23 + 4/60, "Amreli": 21 + 36/60, "Okha": 22 + 29/60}
SEASON_BY_MONTH = {
    1: "Winter", 2: "Winter", 3: "Pre-Monsoon", 4: "Pre-Monsoon", 5: "Pre-Monsoon",
    6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
    10: "Post-Monsoon", 11: "Post-Monsoon", 12: "Post-Monsoon",
}
PERIODS = ["Annual", "Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]
MIN_DAYS = 30
A_AP, B_AP = 0.25, 0.50
GSC = 0.0820  # conventional FAO-56 unit MJ m-2 min-1
DR_COEFFICIENT = 0.003  # primary paper Eq. 2, p. 5; 0.033 is retained only as sensitivity
EXPECTED_FILES = {f"{station.upper()}.CSV" for station in STATIONS}


## 2. Authoritative model registry

Exact source equations are separated from implemented equations where necessary. All 16 IDs are defined here, in one notebook registry.

In [2]:
SOURCE_MODEL_REGISTRY = [
    {"model": "M1", "author": "Hargreaves & Samani (1985), used as-is", "equation": "Rs = 0.16 * sqrt(Tmax - Tmin) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M2", "author": "Rao et al. (2017)", "equation": "Rs = 3.6 * (1.29 + 0.3626 * (Tmax - Tmin))", "inputs": "Tmax, Tmin", "units": "Tmax/Tmin °C; output treated as MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "unit interpretation unresolved", "assumption": "The paper omits units; factor 3.6 interprets the printed output as kWh m-2 d-1 before comparison in MJ. Requires confirmation from the original Rao paper."},
    {"model": "M3", "author": "Richardson (2018), recalibrated by Marif et al. (2022)", "equation": "Rs = 0.3124 * (Tmax - Tmin)^0.2942 * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M4", "author": "Sarkar & Sifat (2016), recalibrated by Marif et al. (2022)", "equation": "Rs = (0.7730 - 0.0296*dT + 0.0016*dT^2) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M5", "author": "Chen et al. (2004), recalibrated by Marif et al. (2022)", "equation": "Rs = (0.2063 + 0.1801*ln(dT)) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M6", "author": "Hargreaves & Riley (1985), recalibrated by Marif et al. (2022)", "equation": "Rs = (0.3040 + 0.1008*sqrt(dT)) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M7", "author": "Panday & Katiyar (2010)", "equation": "Rs = (0.2889 + 0.1562*Tmax/Tmin) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "Requires non-zero Tmin; such rows must yield finite estimates."},
    {"model": "M8", "author": "Falayi et al. (2008)", "equation": "Rs = (1.7217 - 1.691*Theta) * Ra", "inputs": "Theta, Ra; implementation uses Tmin/Tmax", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "Theta undefined in paper", "assumption": "Theta is interpreted as Tmin/Tmax, by analogy with M12; the supplied paper does not define it."},
    {"model": "M9", "author": "Chen et al. (2004), used as-is", "equation": "Rs = (0.28*ln(dT) + 0.15) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M10", "author": "Ertekin & Yaldiz (1999), used as-is", "equation": "Rs = -4.46 + 0.477*Ra - 0.226*Tmean", "inputs": "Tmean, Ra", "units": "Tmean °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "Tmean = (Tmax + Tmin)/2."},
    {"model": "M11", "author": "Sivamadhavi & Selvaraj (2012)", "equation": "Rs = (0.4610 + 0.0029*dT) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M12", "author": "Sivamadhavi & Selvaraj (2012)", "equation": "Rs = (0.5290 + 0.0516*Theta) * Ra; Theta = Tmin/Tmax", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None; Theta is defined in the paper as Tmin/Tmax."},
    {"model": "M13", "author": "Hassan et al. (2016), recalibrated by Ghazouani et al. (2022)", "equation": "Rs = (0.206590 + 0.026851*Tmax - 0.024184*Tmin) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M14", "author": "Kirmani et al. (2015)", "equation": "Rs = (0.85235 - 0.00997*Tmean) * Ra", "inputs": "Tmean, Ra", "units": "Tmean °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "Tmean = (Tmax + Tmin)/2."},
    {"model": "M15", "author": "Hargreaves & Samani (1982), recalibrated by Onyeka et al. (2021)", "equation": "Rs = (0.236 + 0.106*sqrt(dT)) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, p. 7", "status": "as printed", "assumption": "None"},
    {"model": "M16", "author": "Samani (2000), used as-is", "equation": "Rs = (0.00185*dT^2 - 0.0433*dT + 0.4023) * Ra", "inputs": "Tmax, Tmin, Ra", "units": "Tmax/Tmin °C; Ra/Rs MJ m-2 d-1", "source": "Karale et al. (2026) preprint, Table 2, pp. 7-8", "status": "as printed", "assumption": "The supplied study prints this form without sqrt(dT); no alternate form is substituted."},
]

PUBLISHED_EQUATIONS = {'M1': 'Rs = 0.16*sqrt(Tmax-Tmin)*Ra', 'M2': 'H = 1.29 + 0.3626*(Tmax-Tmin)', 'M3': 'H/H0 = 0.3124*(Tmax-Tmin)^0.2942', 'M4': 'H/H0 = 0.7730 - 0.0296*DeltaT + 0.0016*DeltaT^2', 'M5': 'H/H0 = 0.2063 + 0.1801*ln(DeltaT)', 'M6': 'H/H0 = 0.3040 + 0.1008*sqrt(DeltaT)', 'M7': 'H/H0 = 0.2889 + 0.1562*(Tmax/Tmin)', 'M8': 'H/H0 = 1.7217 - 1.691*Theta', 'M9': 'Rs = (0.28*ln(Tmax-Tmin) + 0.15)*Ra', 'M10': 'Rs = -4.46 + 0.477*Ra - 0.226*Tmean', 'M11': 'H/H0 = 0.4610 + 0.0029*DeltaT', 'M12': 'H/H0 = 0.5290 + 0.0516*Theta; Theta = Tmin/Tmax', 'M13': 'H/H0 = 0.206590 + 0.026851*Tmax - 0.024184*Tmin', 'M14': 'H/H0 = 0.85235 - 0.00997*Tmean', 'M15': 'H/H0 = 0.236 + 0.106*sqrt(DeltaT)', 'M16': 'H/H0 = 0.00185*DeltaT^2 - 0.0433*DeltaT + 0.4023'}
MODEL_NAMES = {'M1': 'Hargreaves–Samani', 'M2': 'Rao linear temperature-range model', 'M3': 'Richardson–Marif power model', 'M4': 'Sarkar–Sifat–Marif quadratic model', 'M5': 'Chen–Marif logarithmic model', 'M6': 'Hargreaves–Riley–Marif square-root model', 'M7': 'Panday–Katiyar Tmax/Tmin model', 'M8': 'Falayi Theta model', 'M9': 'Chen logarithmic model', 'M10': 'Ertekin–Yaldiz linear model', 'M11': 'Sivamadhavi–Selvaraj temperature-range model', 'M12': 'Sivamadhavi–Selvaraj ratio model', 'M13': 'Hassan–Ghazouani Tmax/Tmin model', 'M14': 'Kirmani mean-temperature model', 'M15': 'Hargreaves–Samani–Onyeka square-root model', 'M16': 'Samani quadratic model'}
MODEL_REGISTRY = [
    {**row, "model_name": MODEL_NAMES[row["model"]],
     "published_equation": PUBLISHED_EQUATIONS[row["model"]],
     "implemented_equation": row["equation"],
     "source_document": "Research_References/1. Seasonal_and_regional_evaluation_of_sixteen_temper.pdf",
     "source": "Karale et al. (2026), Table 2, p. 8" if row["model"] == "M16" else "Karale et al. (2026), Table 2, p. 7"}
    for row in SOURCE_MODEL_REGISTRY
]
for row in MODEL_REGISTRY:
    if row["model"] == "M2":
        row["status"] = "conflicting notation; output-unit interpretation unresolved"
        row["assumption"] = ("Primary Table 2 prints H=1.29+0.3626*DeltaT without an output unit; a supplied secondary companion gives H/H0. "
                              "The ×3.6 conversion is retained from the existing implementation only as an explicit assumption. "
                              "A no-conversion sensitivity is reported; confirmation from Rao et al. (2017) is needed.")
    elif row["model"] == "M8":
        row["assumption"] = ("The primary table leaves Theta undefined and the supplied papers do not reproduce the original Falayi equation. "
                              "Implementation interprets Theta as Tmin/Tmax by analogy with the explicitly defined M12 ratio; this requires confirmation.")
MODEL_IDS = [row["model"] for row in MODEL_REGISTRY]

def registry_frame():
    return pd.DataFrame(MODEL_REGISTRY)


### Vectorized model equations

In [3]:
def predict_all(tmax, tmin, ra) -> pd.DataFrame:
    """Evaluate M1-M16 in MJ m-2 d-1 where defined; undefined inputs become NaN."""
    tmax, tmin, ra = (pd.Series(v, dtype=float).reset_index(drop=True) for v in (tmax, tmin, ra))
    dt = (tmax - tmin).where(tmax > tmin)
    tmean = (tmax + tmin) / 2
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        p = pd.DataFrame(index=tmax.index)
        p["M1"] = 0.16 * np.sqrt(dt) * ra
        p["M2"] = 3.6 * (1.29 + 0.3626 * dt)
        p["M3"] = 0.3124 * dt**0.2942 * ra
        p["M4"] = (0.7730 - 0.0296 * dt + 0.0016 * dt**2) * ra
        p["M5"] = (0.2063 + 0.1801 * np.log(dt)) * ra
        p["M6"] = (0.3040 + 0.1008 * np.sqrt(dt)) * ra
        p["M7"] = (0.2889 + 0.1562 * tmax / tmin) * ra
        p["M8"] = (1.7217 - 1.691 * tmin / tmax) * ra
        p["M9"] = (0.28 * np.log(dt) + 0.15) * ra
        p["M10"] = -4.46 + 0.477 * ra - 0.226 * tmean
        p["M11"] = (0.4610 + 0.0029 * dt) * ra
        p["M12"] = (0.5290 + 0.0516 * tmin / tmax) * ra
        p["M13"] = (0.206590 + 0.026851 * tmax - 0.024184 * tmin) * ra
        p["M14"] = (0.85235 - 0.00997 * tmean) * ra
        p["M15"] = (0.236 + 0.106 * np.sqrt(dt)) * ra
        p["M16"] = (0.00185 * dt**2 - 0.0433 * dt + 0.4023) * ra
    return p

## 3. Solar geometry and Ångström–Prescott reference

In [4]:
def solar_geometry(day_of_year, latitude_deg, dr_coefficient=DR_COEFFICIENT):
    """Return maximum sunshine N (h) and extraterrestrial radiation Ra (MJ m-2 d-1)."""
    j = np.asarray(day_of_year, dtype=float)
    phi = np.radians(latitude_deg)
    dr = 1 + dr_coefficient * np.cos(2 * np.pi * j / 365)
    delta = 0.409 * np.sin(2 * np.pi * j / 365 - 1.39)
    ws = np.arccos(-np.tan(phi) * np.tan(delta))
    n = 24 * ws / np.pi
    ra = (24 * 60 / np.pi) * GSC * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws)
    )
    return n, ra

## 4. Metrics, normalization, GPI and rank

In [5]:
def metrics(estimate, reference, r2_kind="paper"):
    estimate, reference = np.asarray(estimate, float), np.asarray(reference, float)
    error = estimate - reference
    sst = np.sum((reference - reference.mean()) ** 2)
    if r2_kind == "paper":
        r2 = 1 - np.sum(error**2) / sst if sst > 0 else np.nan
    elif r2_kind == "pearson":
        r = np.corrcoef(estimate, reference)[0, 1] if len(reference) > 1 else np.nan
        r2 = r * r
    else:
        raise ValueError(r2_kind)
    return {"n": len(reference), "R2": r2, "RMSE": np.sqrt(np.mean(error**2)),
            "MAE": np.mean(np.abs(error)), "MBE": np.mean(error)}

def add_gpi(table, mbe_kind="signed"):
    """Literal paper GPI: min-max scale each indicator; +1 error, -1 R2; rank ties share rank."""
    table = table.copy()
    indicators = ["R2", "RMSE", "MAE", "MBE"]
    table["GPI"] = 0.0
    for name in indicators:
        values = table[name].abs() if name == "MBE" and mbe_kind == "absolute" else table[name]
        span = values.max() - values.min()
        scaled = (values - values.min()) / span if span > 0 else pd.Series(0.5, index=table.index)
        table[f"y_{name}"] = scaled
        table[f"median_y_{name}"] = scaled.median()
        direction = -1 if name == "R2" else 1
        table["GPI"] += direction * (scaled.median() - scaled)
    table["rank"] = table["GPI"].round(10).rank(ascending=False, method="min").astype(int)
    return table.sort_values(["rank", "model"], kind="stable").reset_index(drop=True)

## 5. Input discovery, validation, preprocessing and daily predictions

In [6]:
def load_station(station):
    source = ROOT / "data" / "raw_imd" / f"{station.upper()}.csv"
    raw = pd.read_csv(source, dtype=str)
    required = {"YEAR", "MN", "DT", "MAX", "MIN", "SSH"}
    if not required.issubset(raw.columns):
        raise ValueError(f"{source} lacks required columns {sorted(required - set(raw.columns))}")
    dates = pd.to_datetime(dict(year=pd.to_numeric(raw.YEAR, errors="coerce"),
                                month=pd.to_numeric(raw.MN, errors="coerce"),
                                day=pd.to_numeric(raw.DT, errors="coerce")), errors="coerce")
    df = pd.DataFrame({"Date": dates,
                       "Tmax": pd.to_numeric(raw.MAX.str.strip(), errors="coerce"),
                       "Tmin": pd.to_numeric(raw.MIN.str.strip(), errors="coerce"),
                       "SSH": pd.to_numeric(raw.SSH.str.strip(), errors="coerce")})
    if df.Date.isna().any():
        raise ValueError(f"{station}: invalid date rows need explicit review ({int(df.Date.isna().sum())})")
    if df.Date.duplicated().any():
        raise ValueError(f"{station}: duplicate dates need explicit review ({int(df.Date.duplicated().sum())})")
    return df.sort_values("Date").reset_index(drop=True)

def prepare_station(station, dr_coefficient=DR_COEFFICIENT):
    df = load_station(station)
    df["Year"] = df.Date.dt.year
    df["Season"] = df.Date.dt.month.map(SEASON_BY_MONTH)
    df["dT"] = df.Tmax - df.Tmin
    df["Tmean"] = (df.Tmax + df.Tmin) / 2
    df["N"], df["Ra"] = solar_geometry(df.Date.dt.dayofyear, LATITUDE[station], dr_coefficient=dr_coefficient)
    sunshine_ok = df.SSH.notna() & (df.SSH >= 0) & (df.SSH <= df.N)
    df["Rs_AP"] = np.where(sunshine_ok, (A_AP + B_AP * df.SSH / df.N) * df.Ra, np.nan)
    predictions = predict_all(df.Tmax, df.Tmin, df.Ra)
    df[MODEL_IDS] = predictions
    df["temperature_ok"] = df.Tmax.notna() & df.Tmin.notna() & (df.dT > 0)
    df["all_models_finite"] = np.isfinite(df[MODEL_IDS]).all(axis=1)
    df["used"] = df.temperature_ok & df.Rs_AP.notna() & df.all_models_finite
    reason = np.full(len(df), "included", dtype=object)
    reason[df.Tmax.isna().to_numpy()] = "missing Tmax"
    reason[df.Tmin.isna().to_numpy()] = "missing Tmin"
    reason[(df.Tmax.notna() & df.Tmin.notna() & (df.dT <= 0)).to_numpy()] = "Tmax <= Tmin"
    reason[(df.temperature_ok & df.SSH.isna()).to_numpy()] = "missing sunshine duration"
    reason[(df.temperature_ok & df.SSH.notna() & (df.SSH < 0)).to_numpy()] = "sunshine below zero"
    reason[(df.temperature_ok & df.SSH.notna() & (df.SSH > df.N)).to_numpy()] = "sunshine exceeds day length"
    reason[(df.temperature_ok & df.Rs_AP.notna() & ~df.all_models_finite).to_numpy()] = "one or more models non-finite"
    df["exclusion_reason"] = reason
    df["station"] = station
    return df

def evaluate_station_periods(days, mbe_kind="signed", r2_kind="paper", m2_scale=3.6):
    rows = []
    for period in PERIODS:
        selected = days[days.used] if period == "Annual" else days[days.used & (days.Season == period)]
        if len(selected) < MIN_DAYS:
            for model in MODEL_IDS:
                rows.append({"station": days.station.iloc[0], "period": period, "model": model,
                             "n": len(selected), "R2": np.nan, "RMSE": np.nan, "MAE": np.nan,
                             "MBE": np.nan, "GPI": np.nan, "rank": np.nan,
                             "status": f"insufficient valid days (<{MIN_DAYS})"})
            continue
        current = selected[MODEL_IDS].copy()
        if m2_scale != 3.6:
            current["M2"] = (1.29 + 0.3626 * selected.dT) * m2_scale
        metric_rows = [{"station": days.station.iloc[0], "period": period, "model": m,
                        **metrics(current[m], selected.Rs_AP, r2_kind)} for m in MODEL_IDS]
        table = add_gpi(pd.DataFrame(metric_rows), mbe_kind)
        table["status"] = "ok"
        rows.extend(table.to_dict("records"))
    return pd.DataFrame(rows)

## 6. Figures

In [7]:
def save_figures(metrics_all):
    FIGURES.mkdir(parents=True, exist_ok=True)
    ordered = metrics_all[metrics_all.status == "ok"].copy()
    stations = STATIONS
    models = MODEL_IDS
    # Figure 1: annual ranks.
    annual = ordered[ordered.period == "Annual"].pivot(index="model", columns="station", values="rank").reindex(index=models, columns=stations)
    fig, ax = plt.subplots(figsize=(7.4, 7.0))
    im = ax.imshow(annual.to_numpy(float), cmap="RdYlGn_r", vmin=1, vmax=16, aspect="auto")
    for i in range(len(models)):
        for j in range(len(stations)):
            ax.text(j, i, f"{annual.iloc[i,j]:.0f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(len(stations)), stations); ax.set_yticks(range(len(models)), models)
    ax.set_title("Annual GPI rank (1 = best)"); fig.colorbar(im, ax=ax, label="Rank")
    fig.tight_layout()
    for name in ["annual_ranks.png"]:
        fig.savefig(FIGURES / name, dpi=180)
    plt.close(fig)
    # Figure 2: all seasonal ranks, grouped by station.
    fig, axes = plt.subplots(1, len(stations), figsize=(15, 7), sharey=True)
    for ax, station in zip(axes, stations):
        grid = ordered[ordered.station == station].pivot(index="model", columns="period", values="rank").reindex(index=models, columns=PERIODS)
        im = ax.imshow(grid.to_numpy(float), cmap="RdYlGn_r", vmin=1, vmax=16, aspect="auto")
        for i in range(len(models)):
            for j in range(len(PERIODS)):
                ax.text(j, i, f"{grid.iloc[i,j]:.0f}", ha="center", va="center", fontsize=6.5)
        ax.set_xticks(range(len(PERIODS)), PERIODS, rotation=35, ha="right")
        ax.set_yticks(range(len(models)), models); ax.set_title(station)
    fig.suptitle("GPI rank by station and season")
    fig.subplots_adjust(left=.055, right=.90, bottom=.16, top=.88, wspace=.12)
    cax = fig.add_axes([.925, .18, .018, .64])
    fig.colorbar(im, cax=cax, label="Rank")
    for name in ["seasonal_ranks.png"]:
        fig.savefig(FIGURES / name, dpi=180)
    plt.close(fig)
    # Figure 3: annual R2 and RMSE for all models/sites.
    fig, axes = plt.subplots(1, 2, figsize=(12, 7))
    for ax, col, title, cmap in [(axes[0], "R2", "R²", "RdYlGn"), (axes[1], "RMSE", "RMSE (MJ m⁻² d⁻¹)", "RdYlGn_r")]:
        grid = ordered[ordered.period == "Annual"].pivot(index="model", columns="station", values=col).reindex(index=models, columns=stations)
        values = grid.to_numpy(float)
        lim = max(1.0, float(np.nanmax(np.abs(values)))) if col == "R2" else None
        im = ax.imshow(values, cmap=cmap, vmin=-lim if lim else None, vmax=lim if lim else None, aspect="auto")
        for i in range(len(models)):
            for j in range(len(stations)):
                ax.text(j, i, f"{values[i,j]:.2f}", ha="center", va="center", fontsize=7)
        ax.set_xticks(range(len(stations)), stations); ax.set_yticks(range(len(models)), models); ax.set_title(title)
        fig.colorbar(im, ax=ax)
    fig.suptitle("Annual model performance"); fig.tight_layout()
    for name in ["annual_metrics.png"]:
        fig.savefig(FIGURES / name, dpi=180)
    plt.close(fig)
    # Figure 4: Top-3 frequency over 15 station-period combinations.
    top = ordered[ordered["rank"] <= 3].groupby(["station", "model"]).size().unstack("station", fill_value=0).reindex(index=models, columns=stations, fill_value=0)
    fig, ax = plt.subplots(figsize=(9, 4.8)); top.plot(kind="bar", ax=ax, width=.82)
    ax.set_ylabel("Top-3 appearances (of 5 periods)"); ax.set_xlabel("Model")
    ax.set_title("Top-3 frequency by station"); ax.legend(title="Station", ncols=3)
    fig.tight_layout()
    for name in ["top3_frequency.png"]:
        fig.savefig(FIGURES / name, dpi=180)
    plt.close(fig)

def save_residual_figure(days, metrics_all):
    annual_winners = metrics_all[(metrics_all.period == "Annual") & (metrics_all["rank"] == 1)]
    fig, ax = plt.subplots(figsize=(9, 5))
    for station in STATIONS:
        winner = annual_winners.loc[annual_winners.station == station, "model"].iloc[0]
        d = days[(days.station == station) & days.used]
        ax.hist(d[winner] - d.Rs_AP, bins=50, density=True, histtype="step", linewidth=1.6,
                label=f"{station} · {winner}")
    ax.axvline(0, color="black", linewidth=.8)
    ax.set_xlabel("Model estimate − Ångström–Prescott reference (MJ m⁻² d⁻¹)")
    ax.set_ylabel("Density")
    ax.set_title("Daily residuals for each station's annual rank-1 model")
    ax.legend()
    fig.tight_layout(); fig.savefig(FIGURES / "residuals.png", dpi=180); plt.close(fig)

## Model registry and implementation assumptions

The following registry cell is authoritative. Each row stores model ID/name, source equation as printed in Karale et al. (2026) Table 2, implemented equation, inputs/units, coefficients/source, page, and any assumption. Table 2 is on p. 7; M16 continues on p. 8. `H/H0` forms are implemented as `Rs = (H/H0)×Ra` where Table 2 defines a ratio.

- **M2:** the primary Table 2 prints `H=1.29+0.3626ΔT` without an output unit, while a supplied secondary companion writes the model as `H/H0`. The sources do not establish ×3.6. The existing ×3.6 implementation is retained only as a labeled assumption and compared with no conversion in sensitivity results.
- **M8:** the paper gives `Θ` but does not define it. The implementation uses `Tmin/Tmax` by analogy with M12. This remains an assumption pending the cited original Falayi paper.
- **M16:** used exactly as printed, without adding `sqrt(ΔT)`.

No model is adjusted to improve rankings.


## Evaluation, GPI and ranking

For prediction `P_i` and reference `O_i`, the notebook calculates RMSE, MAE, signed MBE=`mean(P−O)`, and the paper's printed R²=`1−SSE/SST`. The paper's GPI method (§2.3.1, Table 3, pp. 8–9) min–max scales each of R², RMSE, MAE and MBE across the 16 models within each station-period. It computes `GPI_i = Σ α_j(ỹ_j−y_ij)`, where `ỹ_j` is the median scaled value; `α_R²=−1`, and `α_RMSE=α_MAE=α_MBE=+1`. Contributions are equal with no additional weights. Thus higher R² and lower error values contribute positively under the printed signs.

The prose says MBE “closer to 0” is ideal, while the printed `+1` direction for signed MBE can reward lower, more negative bias. The primary results preserve the printed formula; absolute-MBE is a sensitivity. Constant metrics contribute zero (scaled to 0.5, equal to their median). Rank is descending GPI after rounding to 10 decimals; ties share the minimum rank. Pearson-r² and alternative `dr=0.033` are also reported as sensitivities.


## Pipeline outputs

The ordered cells below perform input discovery, parsing/QC, solar geometry, reference calculation, all model predictions, per-period metrics/GPI/ranks, annual and seasonal tables, station comparison, sensitivity analysis, structural validation, CSV export, figure generation, and SHA-256 manifests. All outputs are regenerated by this notebook; no manually edited result CSV is read as an analysis input.


## 7. Discover and prepare the three input stations

In [8]:
RESULTS.mkdir(parents=True, exist_ok=True)
FINAL.mkdir(parents=True, exist_ok=True)
QC.mkdir(parents=True, exist_ok=True)
VALIDATION.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
input_files = sorted((ROOT / "data" / "raw_imd").glob("*.csv"))
found_files = {path.name.upper() for path in input_files}
assert found_files == EXPECTED_FILES, f"Unexpected/missing raw station files: {sorted(found_files ^ EXPECTED_FILES)}"
data = {station: prepare_station(station) for station in STATIONS}
all_days = pd.concat([data[station] for station in STATIONS], ignore_index=True)
metrics_all = pd.concat([evaluate_station_periods(data[station]) for station in STATIONS], ignore_index=True)
print("Input files:", [p.relative_to(ROOT).as_posix() for p in input_files])
print("Station valid days:", {station: int(frame.used.sum()) for station, frame in data.items()})

Input files: ['data/raw_imd/AHMEDABAD.csv', 'data/raw_imd/AMRELI.csv', 'data/raw_imd/OKHA.csv']
Station valid days: {'Ahmedabad': 14802, 'Amreli': 14718, 'Okha': 14474}


## 8. Registry, complete metrics, daily results and coverage/QC exports

In [9]:
registry = registry_frame()
registry.to_csv(RESULTS / "model_registry.csv", index=False)
metrics_all.to_csv(FINAL / "all_metrics.csv", index=False)
metrics_all[metrics_all.period == "Annual"].to_csv(FINAL / "annual_metrics.csv", index=False)
metrics_all[metrics_all.period != "Annual"].to_csv(FINAL / "seasonal_metrics.csv", index=False)
metrics_all[metrics_all.period == "Annual"].to_csv(FINAL / "annual_gpi.csv", index=False)
metrics_all[metrics_all.period != "Annual"].to_csv(FINAL / "seasonal_gpi.csv", index=False)
all_days[["Date", "station", "Season", "Tmax", "Tmin", "dT", "Tmean", "SSH", "N", "Ra", "Rs_AP", "used", "exclusion_reason"] + MODEL_IDS].to_csv(FINAL / "daily_predictions.csv", index=False)

provenance, qc, yearly, period_summary = [], [], [], []
for station, frame in data.items():
    valid = frame[frame.used]
    source_dates = pd.date_range(frame.Date.min(), frame.Date.max(), freq="D")
    provenance.append({"station": station, "source_file": f"data/raw_imd/{station.upper()}.csv",
        "source_records": len(frame), "source_first_date": frame.Date.min().date(),
        "source_last_date": frame.Date.max().date(), "source_unique_dates": frame.Date.nunique(),
        "missing_calendar_dates_in_span": len(source_dates.difference(pd.DatetimeIndex(frame.Date))),
        "usable_daily_records": len(valid), "usable_first_date": valid.Date.min().date(),
        "usable_last_date": valid.Date.max().date(), "analysis_period": f"{valid.Date.min().year}-{valid.Date.max().year}"})
    checks = {"missing Tmax": int(frame.Tmax.isna().sum()), "missing Tmin": int(frame.Tmin.isna().sum()),
        "missing sunshine": int(frame.SSH.isna().sum()),
        "Tmax <= Tmin": int((frame.Tmax.notna() & frame.Tmin.notna() & (frame.dT <= 0)).sum()),
        "sunshine below zero": int((frame.SSH < 0).sum()),
        "sunshine exceeds N": int((frame.SSH.notna() & (frame.SSH > frame.N)).sum()),
        "all models non-finite": int((~frame.all_models_finite).sum()), "included daily records": int(frame.used.sum())}
    qc.extend({"station": station, "check": name, "count": count, "records": len(frame)} for name, count in checks.items())
    for year, group in frame.groupby("Year"):
        yearly.append({"station": station, "year": int(year), "records_in_file": len(group),
            "valid_days_used": int(group.used.sum()),
            "expected_calendar_days": 366 if pd.Timestamp(year=int(year), month=12, day=31).is_leap_year else 365})
    for period in PERIODS:
        selected = valid if period == "Annual" else valid[valid.Season == period]
        period_summary.append({"station": station, "period": period, "valid_days": len(selected),
            "first_valid_date": selected.Date.min().date(), "last_valid_date": selected.Date.max().date()})
pd.DataFrame(provenance).to_csv(QC / "data_provenance.csv", index=False)
pd.DataFrame(qc).to_csv(QC / "qc_report.csv", index=False)
pd.DataFrame(yearly).to_csv(QC / "year_coverage.csv", index=False)
pd.DataFrame(period_summary).to_csv(FINAL / "station_period_summary.csv", index=False)
provenance_frame = pd.DataFrame(provenance)
provenance_frame[["station", "source_records", "source_first_date", "source_last_date", "usable_daily_records", "usable_first_date", "usable_last_date"]]

,station,source_records,source_first_date,source_last_date,usable_daily_records,usable_first_date,usable_last_date
0,Ahmedabad,14973,1985-01-01,2025-12-31,14802,1985-01-01,2025-12-23
1,Amreli,14959,1985-01-01,2025-12-31,14718,1985-01-01,2025-12-31
2,Okha,14958,1985-01-01,2025-12-31,14474,1985-01-01,2025-12-31


## 9. Annual, seasonal, station comparisons and Top-3

In [10]:
top3 = metrics_all[(metrics_all.status == "ok") & (metrics_all["rank"] <= 3)].copy()
ranking = metrics_all.pivot(index=["station", "model"], columns="period", values="rank").reindex(
    index=pd.MultiIndex.from_product([STATIONS, MODEL_IDS], names=["station", "model"]), columns=PERIODS)
ranking.to_csv(FINAL / "model_ranking.csv")
top3.to_csv(FINAL / "top_3_models.csv", index=False)
freq = top3.groupby(["station", "model"]).size().rename("top3_periods").reset_index()
grid = pd.MultiIndex.from_product([STATIONS, MODEL_IDS], names=["station", "model"]).to_frame(index=False)
freq = grid.merge(freq, how="left", on=["station", "model"]).fillna({"top3_periods": 0})
freq["top3_periods"] = freq.top3_periods.astype(int)
total = top3.groupby("model").size().rename("top3_periods_all_stations").reset_index()
freq = freq.merge(total, on="model", how="left").fillna({"top3_periods_all_stations": 0})
freq.to_csv(FINAL / "top3_frequency.csv", index=False)
comparisons = []
for station in STATIONS:
    row = {"station": station, "valid_days": int(data[station].used.sum())}
    for period in PERIODS:
        sub = top3[(top3.station == station) & (top3.period == period)].sort_values(["rank", "model"])
        row[f"Top 3 {period}"] = ", ".join(sub.model.tolist())
    comparisons.append(row)
pd.DataFrame(comparisons).to_csv(FINAL / "station_comparison.csv", index=False)
pd.DataFrame(comparisons).rename(columns={"station": "district", "valid_days": "days used"}).to_csv(FINAL / "district_comparison.csv", index=False)
print("Top-3 models by station and period:")
pd.DataFrame(comparisons)

Top-3 models by station and period:


,station,valid_days,Top 3 Annual,Top 3 Winter,Top 3 Pre-Monsoon,Top 3 Monsoon,Top 3 Post-Monsoon
0,Ahmedabad,14802,"M1, M13, M15","M15, M1, M14","M15, M1, M5","M1, M2, M8","M15, M1, M5"
1,Amreli,14718,"M1, M15, M13","M14, M15, M1","M15, M1, M12","M1, M13, M7","M15, M1, M14"
2,Okha,14474,"M5, M6, M3","M4, M14, M9","M14, M12, M6","M13, M15, M5","M14, M4, M12"


## 10. Sensitivity analysis for unresolved method interpretations

In [11]:
sensitivity_specs = [
    ("primary_method", "primary"),
    ("absolute_MBE", "absolute_mbe"),
    ("Pearson_r_squared", "pearson_r2"),
    ("M2_without_x3_6", "m2_no_conversion"),
    ("alternative_dr_0.033", "dr_0.033"),
]
sensitivity_rows = []
for label, variant in sensitivity_specs:
    if variant == "dr_0.033":
        variant_data = {s: prepare_station(s, dr_coefficient=0.033) for s in STATIONS}
        result = pd.concat([evaluate_station_periods(variant_data[s]) for s in STATIONS], ignore_index=True)
    else:
        kwargs = {"mbe_kind": "absolute"} if variant == "absolute_mbe" else {}
        if variant == "pearson_r2": kwargs = {"r2_kind": "pearson"}
        if variant == "m2_no_conversion": kwargs = {"m2_scale": 1.0}
        result = pd.concat([evaluate_station_periods(data[s], **kwargs) for s in STATIONS], ignore_index=True)
    variant_top = result[(result.status == "ok") & (result["rank"] <= 3)]
    for station in STATIONS:
        for period in PERIODS:
            picked = variant_top[(variant_top.station == station) & (variant_top.period == period)].sort_values(["rank", "model"])
            base = top3[(top3.station == station) & (top3.period == period)].sort_values(["rank", "model"])
            sensitivity_rows.append({"variant": label, "station": station, "period": period,
                "top3": ", ".join(picked.model.tolist()),
                "same_top3_as_primary": set(picked.model) == set(base.model)})
sensitivity_frame = pd.DataFrame(sensitivity_rows)
sensitivity_frame.to_csv(VALIDATION / "sensitivity_analysis.csv", index=False)
sensitivity_frame.groupby("variant").same_top3_as_primary.agg(["sum", "count"]).rename(columns={"sum":"same_top3_sets"})

,same_top3_sets,count
variant,,
M2_without_x3_6,14,15
Pearson_r_squared,5,15
absolute_MBE,5,15
alternative_dr_0.033,15,15
primary_method,15,15


## 11. Structural, metric and reproducibility validation

In [12]:
expected_keys = pd.MultiIndex.from_product([STATIONS, PERIODS, MODEL_IDS], names=["station", "period", "model"])
observed_keys = pd.MultiIndex.from_frame(metrics_all[["station", "period", "model"]])
finite_columns = ["R2", "RMSE", "MAE", "MBE", "GPI", "rank"]
# Independently re-derive the geometry and reference values to guard the full dependency chain.
j = all_days.Date.dt.dayofyear.to_numpy(dtype=float)
phi = np.radians(all_days.station.map(LATITUDE).to_numpy(dtype=float))
dr = 1 + DR_COEFFICIENT * np.cos(2 * np.pi * j / 365)
delta = 0.409 * np.sin(2 * np.pi * j / 365 - 1.39)
ws = np.arccos(-np.tan(phi) * np.tan(delta))
expected_N = 24 * ws / np.pi
expected_Ra = (24 * 60 / np.pi) * GSC * dr * (
    ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws)
)
sunshine_valid = all_days.SSH.notna() & (all_days.SSH >= 0) & (all_days.SSH <= all_days.N)
expected_Rs_AP = (0.25 + 0.50 * all_days.loc[sunshine_valid, "SSH"] / all_days.loc[sunshine_valid, "N"]) * all_days.loc[sunshine_valid, "Ra"]
reference_matches_screenshot = np.allclose(
    all_days.loc[sunshine_valid, "Rs_AP"].to_numpy(dtype=float), expected_Rs_AP.to_numpy(dtype=float),
    rtol=1e-12, atol=1e-12,
)
models_match_temperature_model_registry = all(
    np.allclose(data[s][MODEL_IDS].to_numpy(dtype=float),
                predict_all(data[s].Tmax, data[s].Tmin, data[s].Ra).to_numpy(dtype=float),
                rtol=0, atol=0, equal_nan=True)
    for s in STATIONS
)
checks = [
    ("angstrom_coefficients_match_screenshot", A_AP == 0.25 and B_AP == 0.50),
    ("N_matches_maximum_sunshine_geometry", np.allclose(all_days.N.to_numpy(dtype=float), expected_N, rtol=0, atol=1e-12)),
    ("Ra_matches_primary_study_geometry", np.allclose(all_days.Ra.to_numpy(dtype=float), expected_Ra, rtol=1e-12, atol=1e-12)),
    ("reference_matches_measured_sunshine_equation", reference_matches_screenshot),
    ("M1_M16_remain_temperature_model_predictions", models_match_temperature_model_registry),
    ("exactly_three_stations", set(data) == set(STATIONS)),
    ("exactly_sixteen_models", len(MODEL_IDS) == 16 and set(metrics_all.model) == set(MODEL_IDS)),
    ("five_periods_per_station", set(metrics_all.period) == set(PERIODS)),
    ("all_240_unique_combinations", len(metrics_all) == 240 and observed_keys.is_unique and set(observed_keys) == set(expected_keys)),
    ("all_metrics_gpi_ranks_finite", np.isfinite(metrics_all[finite_columns].to_numpy(dtype=float)).all()),
    ("all_combinations_status_ok", set(metrics_all.status) == {"ok"}),
    ("all_included_model_predictions_finite", np.isfinite(all_days.loc[all_days.used, MODEL_IDS].to_numpy(dtype=float)).all()),
    ("all_included_reference_values_finite", np.isfinite(all_days.loc[all_days.used, "Rs_AP"].to_numpy(dtype=float)).all()),
    ("station_scope_matches_configuration", set(all_days.station) == set(STATIONS)),
    ("gpi_ranks_reproducible_within_kernel", pd.concat([evaluate_station_periods(data[s]) for s in STATIONS], ignore_index=True)["rank"].reset_index(drop=True).equals(metrics_all["rank"].reset_index(drop=True))),
    ("all_station_periods_meet_minimum_days", (pd.DataFrame(period_summary).valid_days >= MIN_DAYS).all()),
]
validation = pd.DataFrame([{"check": name, "result": "PASS" if passed else "FAIL"} for name, passed in checks])
validation.to_csv(VALIDATION / "validation_checks.csv", index=False)
freq.to_csv(FINAL / "model_cross_district_summary.csv", index=False)
if not all(passed for _, passed in checks):
    raise AssertionError(validation[validation.result == "FAIL"].to_dict("records"))
validation

,check,result
0,angstrom_coefficients_match_screenshot,PASS
1,N_matches_maximum_sunshine_geometry,PASS
2,Ra_matches_primary_study_geometry,PASS
3,reference_matches_measured_sunshine_equation,PASS
4,M1_M16_remain_temperature_model_predictions,PASS
5,exactly_three_stations,PASS
6,exactly_sixteen_models,PASS
7,five_periods_per_station,PASS
8,all_240_unique_combinations,PASS
9,all_metrics_gpi_ranks_finite,PASS


## 12. Regenerate figures

In [13]:
save_figures(metrics_all)
save_residual_figure(all_days, metrics_all)
figure_files = sorted((ROOT / "figures").rglob("*.png"))
print(f"Generated {len(figure_files)} PNG figures")
print("Figures directory:", FIGURES.relative_to(ROOT))

Generated 5 PNG figures
Figures directory: figures\final


## 13. Reproducibility fingerprints and final summary

In [14]:
# Hash every exported CSV and generated PNG; this manifest itself is intentionally excluded.
output_files = sorted([p for p in RESULTS.rglob("*.csv")] + [p for p in (ROOT / "figures").rglob("*.png")])
hash_rows = [{"path": p.relative_to(ROOT).as_posix(),
              "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
              "bytes": p.stat().st_size} for p in output_files if p.name != "output_hashes.csv"]
hash_manifest = pd.DataFrame(hash_rows)
hash_manifest.to_csv(VALIDATION / "output_hashes.csv", index=False)
print("Validation:", bool((validation.result == "PASS").all()))
print("Station-period-model rows:", len(metrics_all))
print("Study stations:", ", ".join(STATIONS))
print("Model IDs:", ", ".join(MODEL_IDS))
print("Valid days:", {s: int(data[s].used.sum()) for s in STATIONS})
print("Fingerprint records:", len(hash_manifest))
hash_manifest.head()

Validation: True
Station-period-model rows: 240
Study stations: Ahmedabad, Amreli, Okha
Model IDs: M1, M2, M3, M4, M5, M6, M7, M8, M9, M10, M11, M12, M13, M14, M15, M16
Valid days: {'Ahmedabad': 14802, 'Amreli': 14718, 'Okha': 14474}
Fingerprint records: 24


,path,sha256,bytes
0,figures/final/annual_metrics.png,bbb06820df75692f02ecf9ddeb1ce3f343d00d21024d8a...,171466
1,figures/final/annual_ranks.png,c7672922cc468eedd74954904fcc5e1657fd49d73d691d...,67709
2,figures/final/residuals.png,a72d85ddeb6bccc1f6498279d74da895fada6607c515c6...,54237
3,figures/final/seasonal_ranks.png,a983a32c9f74cae8b04b4161a09e3bac107146f89087c9...,164621
4,figures/final/top3_frequency.png,786c16c29afd0976a59290f8a8b89f65fb9cb8d7f26eae...,48869
